In [ ]:
# 1. Refined FG Stock lookup focusing on Series
def get_fg_stock_by_series(master_row):
    series_val = norm(master_row["Series"])

    # Search in Inventory using Series (Material column)
    # We use 'Material' because in your file it often contains the Series code
    r = best_match(df_inventory, series_val, "", series_col="Material")
    if r is not None and not pd.isna(r["Unrestricted"]):
        return r["Unrestricted"]

    # Search in Sheet10
    r = best_match(df_sheet10, series_val, "", series_col="Material")
    if r is not None and not pd.isna(r["Unrestricted"]):
        return r["Unrestricted"]

    # Search in Sheet4
    r = best_match(df_fg, series_val, "", series_col="Series")
    if r is not None and not pd.isna(r["FG Quantity"]):
        return r["FG Quantity"]

    return 0

# 2. Main Loop with Capacity Constraint and Series-Only Focus
plan = []
machine_backlog = {} 
MAX_MINUTES = 1200 # 20 Hours

# Process only unique Series to save time, or iterate normally if each row is unique demand
for i, row in df_master.iterrows():
    series = row["Series"]
    part = row["Part"] # Kept for the final output label
    
    # 3. Use the refined Series-only stock check
    fg_stock = get_fg_stock_by_series(row)
    dispatch = row["Dispatch"] if not pd.isna(row["Dispatch"]) else 0
    available_fg = fg_stock - dispatch
    
    monthly_req = row['Monthely Requirement-jan'] if not pd.isna(row['Monthely Requirement-jan']) else 0
    daily_demand = monthly_req / 28
    
    # 4. Quantity Logic
    if available_fg < daily_demand:
        planned_qty = daily_demand * 2
    elif daily_demand > 0 and daily_demand <= 50:
        planned_qty = 5 * daily_demand
    else:
        planned_qty = daily_demand
        
    if planned_qty <= 0: continue

    # 5. Machine Lookup (Using Series)
    # Note: Ensure 'Part No.' in your tool list contains the Series ID
    machine_row = best_match(df_tool, series, "", series_col="Part No.")
    if machine_row is None: continue
    machine = machine_row["Machine No."]
    
    # 6. Time Calculation
    time_row = best_match(df_prod, series, "", series_col="Part No.")
    if time_row is None or pd.isna(time_row.get("Part Per Hour")): continue
    
    time_hrs = planned_qty / time_row["Part Per Hour"]
    time_mins = time_hrs * 60
    
    # 7. Capacity Filter
    current_used = machine_backlog.get(machine, 0)
    if current_used + time_mins <= MAX_MINUTES:
        machine_backlog[machine] = current_used + time_mins
        plan.append({
            "Machine": machine,
            "Series": series,
            "Part": part,
            "Qty": int(round(planned_qty)),
            "Time_Required_Hrs": round(time_hrs, 2)
        })